In [ ]:
# ============================================================
# Colab: Mann–Kendall (PC1) por município + estatísticas (mean/median/std)
# Saída: projects/YOUR_PROJECT_2/assets/RSEI_TRENDS_PC1
# ============================================================

# -*- coding: utf-8 -*-

import ee
import os
import sys
import copy
import re

# Initialize a map object.
import ee
# Trigger the authentication flow.
ee.Authenticate()
# Initialize the library.
# ee.Initialize(project=os.environ.get('EE_PROJECT'))
ee.Initialize(project=os.environ.get('EE_PROJECT'))
import geemap.core as geemap
# Cloud authentication.
from google.colab import auth
auth.authenticate_user()

In [ ]:
# =========================
# 1) PARÂMETROS DO USUÁRIO
# =========================
asset_indices = 'projects/nexgenmap/MapBiomas2/LANDSAT/DEGRADACAO/mosaics-ndfia'
asset_city    = 'projects/YOUR_PROJECT/assets/CAP_JOIN2'

# Coleção de saída (ImageCollection)
asset_out_root = 'projects/YOUR_PROJECT_2/assets/RSEI_TRENDS_PC1_GAP'

# Lista de municípios (exemplo usa Brasília; adicione os que quiser)
MUNICIPIOS = [
  'Porto Velho',
  #'Rio Branco','Manaus','Boa Vista','Belém',
  'Macapá','Palmas','São Luís','Teresina',
  'Fortaleza','Natal','João Pessoa','Recife','Maceió','Aracaju','Salvador','Belo Horizonte',
  'Vitória','Rio de Janeiro','São Paulo','Curitiba','Florianópolis','Porto Alegre','Campo Grande',
  'Cuiabá','Goiânia','Brasília'
]

# ===============
# 2) UTILITÁRIOS
# ===============
def sanitize_id(name: str) -> str:
    return re.sub(r'[^A-Za-z0-9_]', '_', name)

def ensure_image_collection(ic_asset_id: str):
    """Garante que exista uma ImageCollection no asset indicado."""
    try:
        info = ee.data.getAsset(ic_asset_id)
        if info and info.get('type') == 'IMAGE_COLLECTION':
            return  # já existe
        elif info:
            print(f"[AVISO] Asset existe mas não é ImageCollection: {ic_asset_id} (type={info.get('type')}).")
            return
    except Exception:
        pass
    try:
        ee.data.createAsset({'type': ee.data.ASSET_TYPE_IMAGE_COLL}, ic_asset_id)
        print(f"[OK] ImageCollection criada: {ic_asset_id}")
    except Exception as e:
        print(f"[ERRO] Falha ao criar ImageCollection {ic_asset_id}: {e}")

def temporal_gapfill_ic_bidirectional(ic, band_name='PC1'):
    """Preenche NoData temporalmente (backward+forward) e retorna IC em ordem ASC."""
    ic = ic.select([band_name])
    ic_asc  = ic.sort('system:time_start')
    size    = ic_asc.size()
    ic_desc = ic_asc.sort('system:time_start', False)
    list_desc = ic_desc.toList(size)

    init_acc = ee.Dictionary({'prev': ee.Image(0).updateMask(ee.Image(0)), 'list': ee.List([])})

    def _step_backward(el, acc):
        acc  = ee.Dictionary(acc)
        prev = ee.Image(acc.get('prev'))
        img  = ee.Image(el)
        filled = img.unmask(prev).copyProperties(img, img.propertyNames())
        return ee.Dictionary({'prev': filled, 'list': ee.List(acc.get('list')).add(filled)})

    back_dict = ee.Dictionary(ee.List(list_desc).iterate(_step_backward, init_acc))
    list_back_asc = ee.List(back_dict.get('list')).reverse()
    ic_back_asc = ee.ImageCollection.fromImages(list_back_asc)

    list_fwd = ic_back_asc.toList(size)
    init_acc_fwd = ee.Dictionary({'prev': ee.Image(0).updateMask(ee.Image(0)), 'list': ee.List([])})

    def _step_forward(el, acc):
        acc  = ee.Dictionary(acc)
        prev = ee.Image(acc.get('prev'))
        img  = ee.Image(el)
        filled = img.unmask(prev).copyProperties(img, img.propertyNames())
        return ee.Dictionary({'prev': filled, 'list': ee.List(acc.get('list')).add(filled)})

    fwd_dict = ee.Dictionary(ee.List(list_fwd).iterate(_step_forward, init_acc_fwd))
    list_filled = ee.List(fwd_dict.get('list'))
    ic_filled = ee.ImageCollection.fromImages(list_filled)
    return ic_filled.sort('system:time_start').select([band_name])

def normalize_ic_01(ic, band_name, region, scale=30):
    """Normaliza (0–1) a IC usando min/max globais na região."""
    min_img = ic.select(band_name).min()
    max_img = ic.select(band_name).max()

    minv = ee.Number(ee.Dictionary(min_img.reduceRegion(
        reducer=ee.Reducer.min(), geometry=region, scale=scale, maxPixels=1e13, bestEffort=True
    )).get(band_name, 0))

    maxv = ee.Number(ee.Dictionary(max_img.reduceRegion(
        reducer=ee.Reducer.max(), geometry=region, scale=scale, maxPixels=1e13, bestEffort=True
    )).get(band_name, 1))

    rng  = maxv.subtract(minv)
    safe_rng = ee.Number(ee.Algorithms.If(rng.gt(0), rng, 1))
    minImg   = ee.Image.constant(minv)
    safeRImg = ee.Image.constant(safe_rng)

    def _map_norm(img):
        img  = ee.Image(img)
        norm = img.select(band_name).subtract(minImg).divide(safeRImg).rename(band_name)
        return norm.copyProperties(img, ['system:time_start'])

    return {'collection': ee.ImageCollection(ic.map(_map_norm)), 'min': minv, 'max': maxv}



In [ ]:
# ======================================================
# 3) Função principal: MK + estatísticas por município
# ======================================================
def mk_for_municipio(municipio: str):
    """
    MK (S, Z, significant) sobre PC1 normalizada (0–1) + estatísticas (mean/median/stdDev)
    sobre PC1 original preenchida. Exporta 6 bandas:
      ['significant','Z','S','mean','median','stdDev'].
    mean/median/stdDev são salvas como float.
    """
    try:
        # Geometria do município
        fc_city   = ee.FeatureCollection(asset_city).filter(ee.Filter.eq('NM_MUN', municipio))
        geom_city = fc_city.geometry()

        # IC por município
        ic_raw = ee.ImageCollection(asset_indices) \
                    .filter(ee.Filter.eq('NM_MUN', municipio)) \
                    .sort('system:time_start')

        n_info = ic_raw.size().getInfo()
        if not n_info or n_info < 2:
            return {'municipio': municipio, 'status': 'pulado', 'msg': f'{n_info} imagem(ns); precisa de >= 2.'}

        bands0 = ee.Image(ic_raw.first()).bandNames().getInfo()
        if 'PC1' not in bands0:
            return {'municipio': municipio, 'status': 'erro', 'msg': "Banda 'PC1' não encontrada na coleção."}

        # Seleciona PC1 e recorta
        pc1_ic = ic_raw.select(['PC1']).map(lambda img: ee.Image(img).clip(geom_city))

        # Gap-fill temporal antes de tudo
        pc1_filled_ic = temporal_gapfill_ic_bidirectional(pc1_ic, 'PC1')

        # Normalização 0–1 para MK
        norm_res = normalize_ic_01(pc1_filled_ic, 'PC1', geom_city, scale=30)
        ic_norm  = ee.ImageCollection(norm_res['collection'])

        # ===== Mann–Kendall (S) =====
        time_filter   = ee.Filter.lessThan(leftField='system:time_start', rightField='system:time_start')
        save_all_join = ee.Join.saveAll(matchesKey='after', ordering='system:time_start', ascending=True)
        joined        = save_all_join.apply(ic_norm, ic_norm, time_filter)

        def sign(i, j):
            i = ee.Image(i)
            j = ee.Image(j)
            return j.neq(i).multiply(j.subtract(i).clamp(-1, 1)).int()

        kendall = ee.ImageCollection(
            joined.map(lambda current:
                ee.ImageCollection.fromImages(ee.List(ee.Image(current).get('after'))).map(
                    lambda image: sign(ee.Image(current), ee.Image(image)).unmask(0)
                )
            ).flatten()
        ).reduce('sum', 2).rename('S').clip(geom_city)

        n_num = ee.Number(n_info)
        kendall_variance = n_num.multiply(n_num.subtract(1)).multiply(n_num.multiply(2).add(5)).divide(18)

        zero = kendall.select('S').multiply(kendall.select('S').eq(0))
        pos  = kendall.select('S').multiply(kendall.select('S').gt(0)).subtract(1)
        neg  = kendall.select('S').multiply(kendall.select('S').lt(0)).add(1)

        z = zero.add(pos.divide(kendall_variance.sqrt())).add(neg.divide(kendall_variance.sqrt())) \
                 .rename('Z').clip(geom_city)

        # p e significância (p<=0.025, bicaudal)
        def ee_cdf(z_img):
            return ee.Image(0.5).multiply(ee.Image(1).add(ee.Image(z_img).divide(ee.Image(2).sqrt()).erf()))
        p = ee.Image(1).subtract(ee_cdf(z.abs())).rename('p').clip(geom_city)
        significant = p.lte(0.025).rename('significant').clip(geom_city)  # 0/1

        # ===== Estatísticas (PC1 original preenchida) =====
        mean_img   = pc1_filled_ic.reduce(ee.Reducer.mean()).rename(['mean']).toFloat().clip(geom_city)
        median_img = pc1_filled_ic.reduce(ee.Reducer.median()).rename(['median']).toFloat().clip(geom_city)
        std_img    = pc1_filled_ic.reduce(ee.Reducer.stdDev()).rename(['stdDev']).toFloat().clip(geom_city)

        # ===== Empacota as bandas =====
        out_img = ee.Image.cat([
            significant.toByte(),  # troque para .toFloat() se quiser tudo float
            z.toFloat(),
            kendall.toFloat(),
            mean_img.toFloat(),
            median_img.toFloat(),
            std_img.toFloat()
        ]).rename(['significant','Z','S','mean','median','stdDev']).clip(geom_city)

        # ===== Propriedades seguras =====
        # Evita copiar TODAS as propriedades da origem (pode causar erro de serialização)
        start_date = ee.Date(ee.Image(pc1_filled_ic.first()).get('system:time_start'))
        end_date   = ee.Date(ee.Image(pc1_filled_ic.sort('system:time_start', False).first()).get('system:time_start'))

        out_img = ee.Image(out_img).set({
            'NM_MUN': str(municipio),
            'mk:first_band': 'PC1',
            'mk:n_images': n_num,                # ee.Number é OK
            'mk:p_threshold': 0.025,
            'mk:variance': kendall_variance,     # ee.Number é OK
            'mk:start': start_date.format('YYYY-MM-dd'),
            'mk:end': end_date.format('YYYY-MM-dd'),
            'mk:asset_indices': asset_indices,
            'mk:asset_city': asset_city,
            'mk:description': 'MK (S,Z,significant) sobre PC1 norm + mean/median/stdDev sobre PC1 preenchida'
        })

        # ===== Export =====
        muni_safe    = sanitize_id(municipio)
        asset_id_out = f"{asset_out_root}/MK_{muni_safe}"

        task = ee.batch.Export.image.toAsset(
            image = ee.Image(out_img),     # garante tipo ee.Image
            description = f"MK_{muni_safe}",
            assetId = asset_id_out,
            region = geom_city,            # ee.Geometry é aceito
            scale = 30,
            maxPixels = 1e13
        )

        return {'municipio': municipio, 'status': 'ok', 'n_images': n_info, 'asset_out': asset_id_out, 'task': task}

    except Exception as e:
        return {'municipio': municipio, 'status': 'erro', 'msg': str(e)}

In [ ]:
# ============================
# 4) Garante IC de saída e run
# ============================
ensure_image_collection(asset_out_root)

results = []
for m in MUNICIPIOS:
    info = mk_for_municipio(m)
    results.append(info)
    if info.get('status') == 'ok':
        print(f"[OK] Task criada: {info['municipio']} -> {info['asset_out']}")
        if 'task' in info and info['task'] is not None:
            info['task'].start()
        else:
            print(f"[ERRO] Task não encontrada no resultado para {info['municipio']}")
    else:
        print(f"[{info.get('status','erro').upper()}] {m}: {info.get('msg','')}")

# Resumo
ok_count   = sum(1 for r in results if r.get('status') == 'ok')
skip_count = sum(1 for r in results if r.get('status') == 'pulado')
err_count  = sum(1 for r in results if r.get('status') not in ('ok','pulado'))

print("\n==================== RESUMO ====================")
print(f"Tasks OK criadas: {ok_count}")
print(f"Pulados (<2 imgs): {skip_count}")
print(f"Erros: {err_count}")
for r in results:
    print(f" - {r['municipio']}: {r.get('status')} ({r.get('msg','')})")

print("\nDica: monitore as tasks com:")
print("for t in ee.batch.Task.list():\n    print(t.status())")